# Aziz Tahin: ücretsiz video atölyesi (Google Colab)

Bu defter üç iş yapar, hepsi ücretsiz:
1. **Fotoğraf temizleme:** naylon, poşet, kova, kâğıt gibi istenmeyen şeyleri fotoğraftan siler (LaMa).
2. **Fotoğraftan video:** fotoğrafı 3-4 saniyelik gerçekçi bir klibe çevirir (LTX-Video).
3. **Türkçe seslendirme:** metni erkek sesiyle MP3 yapar (edge-tts).

Çıkan klipleri ve sesi Claude'a gönderin; kurgu, yazılar ve son kart HyperFrames ile orada yapılır.

**Başlamadan:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin, kaydedin.
Hücreleri yukarıdan aşağı sırayla ▶ ile çalıştırın.

## 0. Ekran kartı var mı?
Altta `Tesla T4` yazmalı. Yazmıyorsa yukarıdaki T4 GPU adımını yapın.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Kurulum (1-2 dakika)

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg edge-tts
!pip install -q --no-deps simple-lama-inpainting
import os, gc, glob, torch, numpy as np
from PIL import Image, ImageFilter
os.makedirs("ciktilar", exist_ok=True)
print("Kurulum tamam")

## 2. Fotoğrafları yükleyin
Makine, dükkan içi, raflar... ne kullanacaksanız seçin. Birden fazla seçebilirsiniz.

In [ ]:
from google.colab import files
yuklenen = files.upload()
fotolar = list(yuklenen.keys())
for i, f in enumerate(fotolar):
    print(i, f, Image.open(f).size)

## 3. Fotoğraf temizleme (naylonları kaldırma)

`SILINECEKLER` listesine fotoğraftan silinmesini istediğiniz şeyleri **İngilizce** yazın.
Model bunları otomatik bulur, kırmızıyla gösterir, sonra siler.

- Kırmızı alan eksik kalırsa `ESIK` değerini düşürün (ör. 0.3), fazla yere taşarsa yükseltin (ör. 0.5).
- **Dürüst not:** Makinenin üstünü kaplayan büyük naylonlarda LaMa altını tahmin etmek zorunda kalır; metal yüzey
  bulanık çıkabilir. O zaman bu adımı Gemini uygulamasında yapıp temiz fotoğrafı 4. adıma yükleyin.

In [ ]:
FOTO = fotolar[0]          # hangi fotoğraf: listedeki numarayı yazın
SILINECEKLER = ["plastic bag", "plastic wrap", "plastic sheet", "white bucket", "hose", "cable", "paper sign"]
ESIK = 0.4

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from simple_lama_inpainting import SimpleLama

img = Image.open(FOTO).convert("RGB")
proc = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
seg = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")
inp = proc(text=SILINECEKLER, images=[img] * len(SILINECEKLER), padding=True, return_tensors="pt")
with torch.no_grad():
    logits = seg(**inp).logits
heat = torch.sigmoid(logits).reshape(len(SILINECEKLER), *logits.shape[-2:]).max(0).values.numpy()
maske = Image.fromarray(((heat > ESIK) * 255).astype(np.uint8)).resize(img.size)
maske = maske.filter(ImageFilter.MaxFilter(31))   # kenarları biraz genişlet

goster = img.copy()
goster.paste(Image.new("RGB", img.size, (255, 0, 0)), mask=maske.point(lambda v: 110 if v else 0))
display(goster.resize((img.width * 512 // img.height, 512)))

In [ ]:
lama = SimpleLama()
temiz = lama(img, maske.convert("L"))
temiz = temiz.resize(img.size)
temiz_ad = "ciktilar/temiz_" + os.path.splitext(os.path.basename(FOTO))[0] + ".jpg"
temiz.save(temiz_ad, quality=95)
yan = Image.new("RGB", (img.width * 2, img.height)); yan.paste(img, (0, 0)); yan.paste(temiz, (img.width, 0))
display(yan.resize((yan.width * 512 // yan.height, 512)))
print("Kaydedildi:", temiz_ad)
del seg, proc, lama; gc.collect(); torch.cuda.empty_cache()

## 4. Fotoğraftan video (LTX-Video)

- İlk çalıştırmada model indirilir (~15 GB, 5-10 dk). Bir klip T4'te yaklaşık 10-20 dakika sürer.
- `GIRIS` olarak temizlenmiş fotoğrafı (`ciktilar/temiz_...jpg`) ya da yüklediğiniz herhangi bir fotoğrafı yazın.
- Komutu **İngilizce** yazın: ne hareket etsin, kamera ne yapsın. Fotoğrafta olmayan şeyi istemeyin.
- Hafıza hatası (`CUDA out of memory`) alırsanız `KARE` değerini 49'a düşürün.

In [ ]:
GIRIS = sorted(glob.glob("ciktilar/temiz_*.jpg"))[-1] if glob.glob("ciktilar/temiz_*.jpg") else fotolar[0]
KOMUT = ("The tahini grinding machine is running, the top wheel slowly turns, smooth glossy tahini flows from the spout, "
         "warm light reflecting on stainless steel, slow cinematic push-in, realistic documentary footage")
OLUMSUZ = "worst quality, inconsistent motion, blurry, jittery, distorted, text, watermark, people"
GENISLIK, YUKSEKLIK = 512, 768   # dikey; 32'nin katı olmalı
KARE = 65                         # 8'in katı + 1 (65 kare ≈ 2.6 sn, 97 kare ≈ 3.9 sn)
ADIM = 30
TOHUM = 7
print("Giriş:", GIRIS)

In [ ]:
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

MODEL = "Lightricks/LTX-Video"
DTYPE = torch.bfloat16

# a) Yalnızca metin kodlayıcı: komutu sayıya çevir, sonra hafızadan at
from transformers import T5EncoderModel
# metin kodlayıcı doğrudan ekran kartına yüklenir; Colab'ın 12 GB RAM'i dolmasın
te = T5EncoderModel.from_pretrained(MODEL, subfolder="text_encoder", torch_dtype=DTYPE, device_map="cuda")
p = LTXImageToVideoPipeline.from_pretrained(MODEL, text_encoder=te, transformer=None, vae=None, torch_dtype=DTYPE)
with torch.no_grad():
    pe, pm, ne, nm = p.encode_prompt(KOMUT, OLUMSUZ, do_classifier_free_guidance=True, device="cuda")
del p, te; gc.collect(); torch.cuda.empty_cache()

# b) Video modeli
pipe = LTXImageToVideoPipeline.from_pretrained(MODEL, text_encoder=None, tokenizer=None, torch_dtype=DTYPE)
pipe.to("cuda")
pipe.vae.enable_tiling()
print("Model hazır")

In [ ]:
def kirp(im, w, h):
    # fotoğrafı videonun oranına ortadan kırp
    im = im.convert("RGB"); r = w / h
    if im.width / im.height > r:
        nw = int(im.height * r); l = (im.width - nw) // 2; im = im.crop((l, 0, l + nw, im.height))
    else:
        nh = int(im.width / r); t = (im.height - nh) // 2; im = im.crop((0, t, im.width, t + nh))
    return im.resize((w, h), Image.LANCZOS)

kare0 = kirp(Image.open(GIRIS), GENISLIK, YUKSEKLIK)
with torch.no_grad():
    video = pipe(image=kare0, prompt_embeds=pe, prompt_attention_mask=pm,
                 negative_prompt_embeds=ne, negative_prompt_attention_mask=nm,
                 width=GENISLIK, height=YUKSEKLIK, num_frames=KARE, num_inference_steps=ADIM,
                 decode_timestep=0.03, decode_noise_scale=0.025,
                 generator=torch.Generator("cuda").manual_seed(TOHUM)).frames[0]
ad = f"ciktilar/klip_{len(glob.glob('ciktilar/klip_*.mp4')) + 1}.mp4"
export_to_video(video, ad, fps=25)
from IPython.display import Video
display(Video(ad, embed=True, width=300))
print("Kaydedildi:", ad)

**Başka bir klip için:** 4. adımdaki ayar hücresinde `GIRIS` ve `KOMUT`u değiştirin; sonra **(a)/(b) model hücresini de**
yeniden çalıştırın (komut değişince yeniden kodlanması gerekiyor), ardından son hücreyi çalıştırın.
Beğenmediğiniz klip için yalnızca `TOHUM` sayısını değiştirip son hücreyi tekrar çalıştırmanız yeterli.

## 5. Türkçe seslendirme

Microsoft Edge'in ücretsiz seslerini kullanır. Erkek sesler: `tr-TR-AhmetNeural`. Kadın: `tr-TR-EmelNeural`.
Not: Bu servis ücretsiz ama Microsoft'un kullanım koşullarına tabidir; reklamda kullanmadan önce koşullara bir göz atın.

In [ ]:
import edge_tts
METIN = ("Bursa'nın tarihi çarşısında, her sabah aynı heyecanla açarız. "
         "Raflarımızda Aziz Tahin, en taze haliyle sizi bekler. "
         "Tahinimizi kendi makinemizde öğütürüz. Yüzde yüz susam; şeker yok, koruyucu yok. "
         "Siparişinizi özenle paketler, Türkiye'nin her iline göndeririz. "
         "İçine hiçbir şey eklemeyiz. Yalnızca susam. "
         "Aziz Tahin. Sipariş için bize DM'den yazın.")
SES = "tr-TR-AhmetNeural"
HIZ = "-8%"     # daha yavaş için "-15%", daha hızlı için "+0%"
await edge_tts.Communicate(METIN, SES, rate=HIZ).save("ciktilar/seslendirme.mp3")
from IPython.display import Audio
display(Audio("ciktilar/seslendirme.mp3"))

## 6. Hepsini indir
Tek bir `ciktilar.zip` iner. Bu dosyayı Claude sohbetine gönderin.

In [ ]:
!cd ciktilar && zip -q -r ../ciktilar.zip . && cd .. && ls -la ciktilar
files.download("ciktilar.zip")